# 11b — Measuring the machine

ch10 reported "~1.8 ms per frame" and hypothesized where it went. A
hypothesis is not an instrument. This half-step (10b, inserted at the ch10
walkthrough) builds the measurement tools the next steps will lean on —
step 11's ray-march verdict and ch15's four-target comparison must not rest
on un-decomposed wall clock.

Three tools, all satellites (the kernel was finished last chapter and was
not touched):

- **`bench.benchmark`** — BenchmarkTools-style adaptive sampling: warm up,
  TUNE evals-per-sample above the timer's resolution floor, then sample to a
  budget. The **minimum** is the estimator (noise on a quiet machine is
  strictly additive).
- **`bench.instrument`** — phase decomposition of the hot path, riding the
  kernel's EVENT SEAM (design 120): arming a sink routes `dispatch` through
  a traced twin that stamps every phase. (Historical note: this shipped at
  step 10b as a monkeypatch on `FastRecord` fields because the kernel budget
  had 3 lines of headroom; the budget conversation that followed is design
  120 §9 — the seam replaced the surgery.)
- **`bench.gpu_timeline`** — WebGPU `timestamp-query` begin/end-of-pass
  clocks (nanoseconds by spec) splitting `launch` into encode+submit / GPU
  execution / readback. The adapter feature is requested by the demo device
  when present.

In [1]:
import pdum.dsl  # noqa: F401
from pdum.dsl import jit, viz
from pdum.dsl.bench import benchmark, instrument, phase_timeline

viz.install()


def make(cx, gain):
    @jit()
    def f(x):
        return gain * (x - cx)

    return f


f = make(0.1, 2.0)
f(1.0)
trial = benchmark(lambda: f(1.0))
print("hot path      :", trial)
rebuild = benchmark(lambda: make(0.1, 2.0)(1.0))
print("rebuild + call:", rebuild)
print()
print(f"the full factory pattern costs {(rebuild.minimum - trial.minimum) * 1e6:.2f} µs of phase A per frame")
print(f"evals/sample was TUNED to {trial.evals} — a naive one-call timing of a")
print(f"~{trial.minimum * 1e6:.0f} µs path would be mostly measuring the clock")

hot path      : Trial(min 2.46 µs · median 2.55 µs · mean 2.61 µs · 2996 samples × 32 evals)


rebuild + call: Trial(min 4.15 µs · median 4.28 µs · mean 4.40 µs · 3549 samples × 16 evals)

the full factory pattern costs 1.69 µs of phase A per frame
evals/sample was TUNED to 32 — a naive one-call timing of a
~2 µs path would be mostly measuring the clock


## Where the microseconds go

`instrument` wraps the warm record's `extract` and `launch` with timestamp
shims for a few frames, then restores them. The phases are the hit path's
anatomy from ch09, now with numbers attached — rendered by the new timeline
widget (static HTML, like every widget in this book):

In [2]:
phases = instrument(f, 1.0, frames=200)
for name in ("key+probe", "extract", "pack", "launch"):
    print(f"  {name:<10} {phases[name] * 1e6:6.2f} µs")
print(f"  {'total':<10} {phases['total'] * 1e6:6.2f} µs")
phase_timeline(phases)

  key+probe    0.93 µs
  extract      0.38 µs
  pack         0.56 µs
  launch       0.40 µs
  total        2.27 µs


Timeline(spans=[('key+probe', 0.0, 9.341800000000005e-07, 'host'), ('extract', 9.341800000000005e-07, 3.756199999999998e-07, 'host'), ('pack', 1.3098000000000004e-06, 5.59219999999999e-07, 'host'), ('launch', 1.8690199999999995e-06, 4.0224500000000033e-07, 'host')], title='dispatch')

## The ch10 question, answered

One 256×256 GPU frame, decomposed. The `gpu` lane is the begin/end-of-pass
timestamp delta — what the silicon actually spent:

In [3]:
from pdum.dsl.bench import gpu_timeline


def gmake(cx):
    @jit(kind="simple_shader.compute")
    def k(i, j):
        return (i / 128.0 - cx) * (j / 128.0)

    return k


tl = gpu_timeline(gmake(0.5), out=(256, 256))
for name, _, dur, lane in tl.spans:
    print(f"  {name:<14} {dur * 1e6:8.1f} µs   [{lane}]")
print(f"  {'frame total':<14} {tl.total * 1e3:8.2f} ms")
tl

  encode+submit     454.5 µs   [host]
  gpu                12.7 µs   [gpu]
  readback         1686.3 µs   [host]
  frame total        2.15 ms


Timeline(spans=[('encode+submit', 0.0, 0.00045449570752680295, 'host'), ('gpu', 0.00045449570752680295, 1.2697999999999998e-05, 'gpu'), ('readback', 0.00046719370752680296, 0.0016863147262483832, 'host')], title='one GPU frame')

The verdict, with instruments instead of adjectives: the GPU computes this
kernel in single-digit **micro**seconds. The milliseconds are the blocking
readback (a full submit→wait→map round trip — the *boundary act* ch08
promised would be the only place bytes cross) plus bridge encode/submit.
A render loop that draws instead of reading back never pays the big span.

## Scaling: what actually moves

In [4]:
for side in (64, 256, 1024):
    tl = gpu_timeline(gmake(0.5), out=(side, side), frames=5)
    spans = {n: d for n, _, d, _ in tl.spans}
    print(
        f"  {side:>4}²  encode {spans['encode+submit'] * 1e6:7.1f} µs   "
        f"gpu {spans['gpu'] * 1e6:7.1f} µs   readback {spans['readback'] * 1e6:8.1f} µs"
    )
print()
print("the surprise: readback barely moves from 64² (16 KB) to 1024² (4 MB) —")
print("it is dominated by FIXED sync latency (submit→wait→map), not bandwidth.")
print("gpu time is micro-scale and quantizes noisily at one-pass resolution.")
print("host phases from the previous section never appear at this magnification.")

    64²  encode    92.9 µs   gpu     9.2 µs   readback   1647.3 µs
   256²  encode   109.8 µs   gpu    11.6 µs   readback   1625.1 µs
  1024²  encode   186.4 µs   gpu    58.2 µs   readback   1961.3 µs

the surprise: readback barely moves from 64² (16 KB) to 1024² (4 MB) —
it is dominated by FIXED sync latency (submit→wait→map), not bandwidth.
gpu time is micro-scale and quantizes noisily at one-pass resolution.
host phases from the previous section never appear at this magnification.


## Things to notice

- The thresholds from the step-9 gate table are now REAL: the suite fails if
  the warm hit path exceeds the fail line (`tests/test_bench.py`), instead
  of a notebook printing a number nobody re-reads.
- `instrument` works on ANY backend and never touches a record — it reads
  the same event stream everything else does. ch15 will point it at four
  targets.
- The timeline widget is ~35 lines of the viz satellite: spans in, static
  HTML out, hover for µs. No scripts, both notebook hosts.

## The event seam: counting the expensive things

Design 120's addition. Every should-be-rare occurrence — a miss, a compile,
a guard drift, an eviction — announces itself on a kernel seam that costs
one list check when dark. `events.record()` turns a block into a report:
exact counts always; stacks and timings sampled; ten thousand events from
one loop interned to ONE stack. The bug class that motivated it — a capture
rebound every frame, recompiling forever with correct answers and no
symptom — is now both visible and assertable:

In [5]:
from pdum.dsl import events


def drifty_pair():
    c = 1.0

    @jit()
    def g(x):
        return c * x

    def rebind(v):
        nonlocal c
        c = v

    return g, rebind


g, rebind = drifty_pair()
g(1.0)
with events.record() as ev:
    for i in range(50):
        rebind(float(i))  # the same cell, new contents, every frame
        g(1.0)
print(ev)
print()
print("the ONE stack behind all of it:", ev["guard.drift"].exemplars[0].trace.user_frames[0])

event                          count     total      mean  traces
dispatch.probe                    50     7.4ms   147.0µs       1
spec.compile                      50     5.9ms   117.1µs       1
dispatch.pack                     50    52.2µs     1.0µs       1
dispatch.launch                   50    35.8µs     716ns       1
dispatch.extract                  50    34.9µs     698ns       1
guard.drift                       50       0ns       0ns       1
spec.miss                         50       0ns       0ns       1
  lower                           50     2.4ms    47.5µs       1
  rewrite                         50     1.4ms    27.0µs       1



the ONE stack behind all of it: <module> (/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83305/2990944662.py:23)


In [6]:
# The assertion you could not write before (120 §6.4): captures are stable.
stable = make(0.5, 2.0)
stable(1.0)
with events.forbid("guard.drift"):
    for _ in range(100):
        stable(1.0)
print("100 frames, zero drift — now a CI-shaped guarantee")


# And the miss path, dark since step 8, is a phase tree:
def novel(c):
    @jit()
    def k(x):
        return sqrt(x * c) + c  # noqa: F821 — a body this notebook has never compiled

    return k


with events.record() as ev2:
    novel(2.5)(1.0)  # one genuinely cold compile
print(ev2)

100 frames, zero drift — now a CI-shaped guarantee
event                          count     total      mean  traces
dispatch.probe                     1   754.8µs   754.8µs       1
spec.compile                       1   634.2µs   634.2µs       1
dispatch.launch                    1     5.0µs     5.0µs       1
dispatch.pack                      1     1.5µs     1.5µs       1
dispatch.extract                   1     958ns     958ns       1
capture.snapshot                   1       0ns       0ns       1
fntype.miss                        1       0ns       0ns       1
spec.miss                          1       0ns       0ns       1
  artifact.compile                 1   224.5µs   224.5µs       1
  lower                            1    86.9µs    86.9µs       1
  rewrite                          1    41.3µs    41.3µs       1
  artifact.miss                    1       0ns       0ns       1
    render                         1    24.8µs    24.8µs       1


## What we can't do yet

Per-op GPU attribution (one timestamp pair per PASS is the WebGPU deal;
finer needs vendor tools); async/double-buffered readback (the fix for the
big span is API-shaped — arrives with the graphics `draw(target)` surface);
CUDA events and Metal `gpuStartTime` (their backends, step 14). **Next:
arrays, `core.for`, and the C backend — step 11, with a ray-march verdict
these instruments can now referee.**